In [7]:
import wrds
import pandas as pd
import numpy as np

def g_cmp(u: str, t: str) -> pd.DataFrame:
    """Retrieves global fundamental data from Compustat.

    Args:
        u: WRDS username.
        t: Target ticker symbol.

    Returns:
        DataFrame containing historical fundamentals.
    """
    d = wrds.Connection(wrds_username=u)
    q = f"""
        SELECT datadate, at, lt, nit, revt
        FROM comp.g_funda
        WHERE gvkey IN (
            SELECT gvkey
            FROM comp.g_security
            WHERE tic = '{t}'
        )
        AND fic = 'AUS'
        AND indfmt = 'INDL'
        AND datafmt = 'STD'
        AND popsrc = 'I'
        AND consol = 'C'
        ORDER BY datadate DESC
    """
    return d.raw_sql(q, date_cols=['datadate']).set_index('datadate')

def g_ibs(u: str, t: str) -> pd.DataFrame:
    """Retrieves consensus estimates from LSEG IBES Summary.

    Args:
        u: WRDS username.
        t: Target ticker symbol.

    Returns:
        DataFrame containing EPS summary estimates.
    """
    d = wrds.Connection(wrds_username=u)
    q = f"SELECT statpers, fpedats, meanest, numest FROM ibes.statsumu_epsus WHERE ticker = '{t}' ORDER BY statpers DESC"
    return d.raw_sql(q, date_cols=['statpers', 'fpedats']).set_index('statpers')

def g_crs(u: str, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves daily stock data from CRSP Version 2.

    Args:
        u: WRDS username.
        p: PERMNO identifier.
        s: Start date (YYYY-MM-DD).
        e: End date (YYYY-MM-DD).

    Returns:
        DataFrame containing daily returns and volumes.
    """
    d = wrds.Connection(wrds_username=u)
    q = f"SELECT dlycaldt, dlyret, dlyvol FROM crsp.dsf_v2 WHERE permno = {p} AND dlycaldt >= '{s}' AND dlycaldt <= '{e}' ORDER BY dlycaldt"
    return d.raw_sql(q, date_cols=['dlycaldt']).set_index('dlycaldt')

def g_evt(u: str, t: str) -> pd.DataFrame:
    """Retrieves key developments for event studies from Capital IQ.

    Args:
        u: WRDS username.
        t: Target ticker symbol.

    Returns:
        DataFrame containing event dates and descriptions.
    """
    d = wrds.Connection(wrds_username=u)
    q = f"""
        SELECT a.announcedate, a.keydeveventtypeid, b.headline
        FROM ciq.wrds_keydev a
        JOIN ciq.ciqkeydev b ON a.keydevid = b.keydevid
        WHERE a.companyid IN (
            SELECT companyid
            FROM ciq.wrds_ticker
            WHERE ticker = '{t}'
        )
        ORDER BY a.announcedate DESC
    """
    return d.raw_sql(q, date_cols=['announcedate']).set_index('announcedate')

In [8]:
usr = "zackienzle1"
tk = "WDS"
p_no = 12345
st = "2014-01-01"
ed = "2024-01-01"

df_cmp = g_cmp(usr, tk)
df_ibs = g_ibs(usr, tk)
df_crs = g_crs(usr, p_no, st, ed)
df_evt = g_evt(usr, tk)

Loading library list...
Done
Loading library list...
Done
Loading library list...
Done
Loading library list...
Done


In [13]:
df_cmp

,at,lt,nit,revt
datadate,,,,


In [10]:
df_ibs

,fpedats,meanest,numest
statpers,,,


In [14]:
df_crs

,dlyret,dlyvol
dlycaldt,,
2014-01-02,-0.018685,4922700.0
2014-01-03,-0.000889,1528500.0
2014-01-06,-0.009402,3119500.0
2014-01-07,0.012441,2732900.0
2014-01-08,0.010008,3419200.0
...,...,...
2023-12-22,-0.00394,1243162.0
2023-12-26,0.006454,1154527.0
2023-12-27,-0.002172,928378.0


In [15]:
df_evt

,keydeveventtypeid,headline
announcedate,,
2026-03-26,41,Woodside Energy Group Ltd Assumes Operational ...
2026-03-25,31,Woodside Energy Expands Operations with Beaumo...
2026-03-18,16,Woodside Energy Group Ltd Appoints Mark Cutifa...
2026-03-17,101,Woodside Energy Appoints Elizabeth Westcott as...
2026-03-17,16,Woodside Energy Appoints Elizabeth Westcott as...
...,...,...
2000-03-31,95,Woodside Petroleum Ltd.(ASX:WPL) added to S&P/...
2000-03-31,95,Woodside Petroleum Ltd.(ASX:WPL) added to S&P/...
2000-03-31,95,Woodside Petroleum Ltd.(ASX:WPL) added to S&P/...
